In [1]:
import cv2
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
# JSON 파일 경로 및 이미지 폴더 경로
train_json_path = './dataset/validation/validation_defected_json'
train_img_path = './dataset/validation/validation_defected_dataset'

# valid_json_path = './dataset/validation/validation_defected_json'
# valid_img_path = './dataset/validation/validation_defected_dataset'

# 타일을 저장할 리스트
defected_img = []       # 바운딩 박스가 포함된 타일
not_defected_img = []   # 바운딩 박스가 포함되지 않은 타일

for idx in os.listdir(train_json_path):
    # json files load
    with open(f'{train_json_path}/{idx}', 'r') as f:
        annotations = json.load(f)
    
    # read image
    image = cv2.imread(f'{train_img_path}/{annotations["image_name"]}', cv2.IMREAD_GRAYSCALE)

    # 이미지 및 바운딩 박스 정보 가져오기
    image_name = annotations['image_name']
    top_x = annotations.get('top_x')
    top_y = annotations.get('top_y')
    bot_x = annotations.get('bot_x')
    bot_y = annotations.get('bot_y')

    # 바운딩 박스의 좌표가 모두 있는지 확인
    has_bbox = top_x is not None and top_y is not None and bot_x is not None and bot_y is not None

    # 타일 크기 및 그리드 사이즈 설정
    grid_size = 10
    tile_size = 512 // grid_size

    # 이미지를 10x10으로 슬라이싱하여 타일로 나누고 업스케일링
    for i in range(grid_size):
        for j in range(grid_size):
            # 타일 슬라이싱
            tile = image[i * tile_size:(i + 1) * tile_size, j * tile_size:(j + 1) * tile_size]
            
            # 타일의 좌표 (원본 이미지에서의 위치)
            tile_top_x = j * tile_size
            tile_top_y = i * tile_size
            tile_bot_x = (j + 1) * tile_size
            tile_bot_y = (i + 1) * tile_size

            # 바운딩 박스와 타일의 겹침 여부 확인
            if has_bbox:
                if not (tile_bot_x < top_x or tile_top_x > bot_x or tile_bot_y < top_y or tile_top_y > bot_y):
                    # 바운딩 박스와 겹치는 타일
                    upscaled_tile = cv2.resize(tile, (512, 512), interpolation=cv2.INTER_LINEAR)
                    defected_img.append(upscaled_tile)
                else:
                    # 겹치지 않는 타일
                    upscaled_tile = cv2.resize(tile, (512, 512), interpolation=cv2.INTER_LINEAR)
                    not_defected_img.append(upscaled_tile)
            else:
                # 바운딩 박스가 없을 경우 모두 not_defected로 분류
                upscaled_tile = cv2.resize(tile, (512, 512), interpolation=cv2.INTER_LINEAR)
                not_defected_img.append(upscaled_tile)

print(len(defected_img))
print(len(not_defected_img))

1129
73871


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, random_split

# CNN 모델 정의
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # 2x2 풀링 레이어
        self.fc1 = nn.Linear(32 * 128 * 128, 128)  # Fully Connected Layer
        self.fc2 = nn.Linear(128, 2)  # Output Layer (2개의 클래스)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 128 * 128)  # Flatten the input
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 이미지 전처리: np.array를 torch.Tensor로 변환
def preprocess_images(defected_img, not_defected_img):
    # 이미지 리스트를 텐서로 변환하고, 각각의 레이블을 붙임
    X_defected = np.array(defected_img)
    X_not_defected = np.array(not_defected_img)
    
    # 라벨 생성: defected는 1, not_defected는 0
    y_defected = np.ones(len(X_defected))  # defected 클래스 라벨 (1)
    y_not_defected = np.zeros(len(X_not_defected))  # not_defected 클래스 라벨 (0)

    # 이미지 데이터와 라벨 결합
    X = np.concatenate([X_defected, X_not_defected], axis=0)
    y = np.concatenate([y_defected, y_not_defected], axis=0)
    
    # 데이터셋을 텐서로 변환
    X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2)  # (batch_size, channels, height, width)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    return X_tensor, y_tensor

# 데이터 로더 생성
def create_data_loader(X, y, batch_size=16):
    dataset = TensorDataset(X, y)
    train_size = int(0.8 * len(dataset))  # 80% 학습, 20% 검증
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# 학습 함수
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cpu'):
    model.to(device)  # 모델을 GPU 또는 CPU로 전송
    for epoch in range(num_epochs):
        # 학습 단계
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # 데이터를 GPU로 전송

            optimizer.zero_grad()  # 기울기 초기화
            outputs = model(inputs)  # 모델에 입력을 전달
            loss = criterion(outputs, labels)  # 손실 계산
            loss.backward()  # 역전파
            optimizer.step()  # 가중치 업데이트
            
            running_loss += loss.item()
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {running_loss / len(train_loader)}')

        # 검증 단계
        model.eval()  # 평가 모드로 전환
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():  # 검증 단계에서는 기울기 계산을 하지 않음
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                # 정확도 계산
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        accuracy = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {val_loss}, Validation Accuracy: {accuracy:.2f}%')

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 중인 장치: {device}")

# 1. defected_img와 not_defected_img는 np.array 형태로 입력됨 (예시로 무작위 데이터 사용)


# 2. 이미지 전처리
X_tensor, y_tensor = preprocess_images(defected_img, not_defected_img)

# 3. 데이터 로더 생성
train_loader, val_loader = create_data_loader(X_tensor, y_tensor)

# 4. 모델, 손실 함수, 옵티마이저 정의
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()  # Cross-Entropy 손실 함수
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam 옵티마이저

# 5. 모델 학습 (GPU 지원)
train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device=device)
